# Baseline: 30 Hz burst vs ARC-EX, four participants

**P03, P04 and P05, cathodic, baseline** - no lidocaine, no vibration, all on **electrode 2**, so
the montage is the same for the three. The question is the same one in each: *does 30 Hz behave
differently from ARC-EX, and does it do so in all three people?*

## The rule this whole notebook follows

**Nothing in mA or mV is compared between participants.** Different anatomy, electrode placement,
impedance and EMG gain make raw numbers meaningless across people. Every figure normalises
*within* a participant first:

| | normalised to |
|---|---|
| intensity | that muscle's own motor threshold (`x MT`) |
| amplitude | that muscle's own biggest response in that recording (`%`) |

What is compared between people is then a **ratio** or a **shape**, never a raw number.

## What comes out

| | figure | the question it answers |
|---|---|---|
| 1 | the four participants in one plot | do the three behave alike once each is normalised to itself? |
| 2 | one figure per muscle, participants side by side | how big is the response, and does it hold up along the train? |

Motor thresholds come first, because everything above depends on them. They are detected
automatically and then **corrected by hand** in the picker, one cell per recording.

> **The intensity rule for this whole project:** every muscle is analysed **at the motor
> threshold you picked for it** in that recording - the trace you chose in the picker is the
> trace the numbers come from. Not a step above it, not one mA for the whole arm. `MT_STEPS = 0`
> everywhere; it exists only so the choice is visible, not so it varies between notebooks.

> **The numbers quoted below were computed with hand-picked thresholds that were lost** (see the
> repo history). Until P03's and P04's recordings are picked again, this notebook falls back to
> automatic detection and the figures will not match them. Re-check the numbers after re-picking -
> the section on motor thresholds prints which recordings are picked and which are detected.


In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from functions import (set_style, load_run, pretty, waterfall,
                       thresholds_for, threshold_source,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       recruitment, fig_threshold_ratio, fig_recruitment,
                       fig_selectivity, fig_muscle_grid,
                       fig_across_subjects, fig_bars,
                       check_thresholds, mt_stamp, warn_if_stale,
                       fig_threshold_grid,
                       fig_detection_row,
                       fig_detection_grid,
                       condition_table, fig_group_summary)
from functions.paper import motor_thresholds, fig_depression, step_up
set_style()

## What this notebook shows

**Cathodic stimulation, electrode 2, baseline, three participants.** P05's two recordings are the
ones the 24-07 log names `burst cathodic` and `modulated cathodic`; P03's session switches to
cathodic on electrode 2 at 15:19 and never switches back; P04's polarity is not written in its log
and was confirmed by the experimenters.

### 1. ARC-EX needs about 2.8x the current of 30 Hz — the solid result

| | P03 | P04 | P05 |
|---|---|---|---|
| ARC-EX threshold / 30 Hz threshold | 2.74 +- 0.17 | 2.88 +- 0.29 | 2.86 +- 0.12 |

Unchanged to two decimals when the response criterion is loosened, tightened to `MIN_SNR = 2.0`,
or made to hold for three intensities instead of two. It is a ratio of two thresholds inside one
participant, so gain, electrode placement and anatomy cancel. Three participants, 4-7 muscles each.

### 2. Recruitment is steep, and the same shape in everyone

Normalised to each muscle's own threshold, the curves are flat to ~0.75 x MT and reach the
threshold response within the last quarter of the range - in both protocols and all three people.
That validates the normalisation; it is not itself a finding. The ramps stop at 1.0-1.2 x MT, so
**supra-threshold recruitment cannot be compared at all** with these recordings.

### 3. Along the train, ARC-EX is lower - but not in everyone

Mean of pulses 2-10, as % of the 1st pulse: ARC-EX is lower than 30 Hz in **8 of 11** muscle
comparisons. A sign test on that is p ~ 0.11, so the headline number is not the point. The
structure is:

| | ARC-EX lower in |
|---|---|
| P04 | 3 of 3 |
| P05 | 4 of 4 |
| **P03** | **1 of 4 - reverses** |

So: consistent within P04 and P05, opposite in P03. P03 is also the session whose baseline was
recorded 1.5 h in with the vibrator already attached, and whose current thresholds give ratios of
5 % - values extreme enough to suggest the threshold is misplaced rather than the muscle unusual.

### What this rests on, and what would settle it

- The thresholds here are **automatically detected**, not picked. Re-pick P03's two baseline
  recordings first: they decide whether point 3 is a result or a threshold artifact.
- P05's responses are small (0.02-0.03 mV in several muscles); its ratios are noise over noise in
  places. The trace row under each bar figure shows which.
- n = 3 participants, 4 muscles, one session each, healthy. Everything above is descriptive.


## 1 · Config

**What this does.** Names the six baseline recordings - three participants x two protocols, all **cathodic**, all electrode 2 - and the eight muscles they share. Prints each one's intensity ladder
so you can see what range there is to work with.

**What we get.** All six recordings have every muscle. The ladders differ a lot: P03 runs to
1.4 x threshold, P04's ARC-EX stops **exactly at** threshold. That single fact limits every
recruitment comparison below.


In [ ]:
# ======== the six recordings ========
RUN = {   # (participant, protocol) -> csv, all cathodic baseline on electrode 2
    ("P03", "burst"): "tSCS_CHUV_data/15-07-2026/P03tscsHealthy/Burst_autosave_20260715_164431_737ms.csv",
    ("P03", "arcex"): "tSCS_CHUV_data/15-07-2026/P03tscsHealthy/Modulated_autosave_20260715_164630_593ms.csv",
    # ^^^ the block the session log (P03_2026-07-15_tendon_vibration.xlsx) marks "baseline":
    #     16:44 burst and 16:46 ARC-EX, recorded before the vibrator was switched on.
    ("P04", "burst"): "tSCS_CHUV_data/16-07-2026/P04tscsHealthy/Burst_autosave_20260716_142311_232ms.csv",
    ("P04", "arcex"): "tSCS_CHUV_data/16-07-2026/P04tscsHealthy/Modulated_autosave_20260716_142536_785ms.csv",
    ("P05", "burst"): "tSCS_CHUV_data/24-07-2026/testSCS/Burst_autosave_20260724_102443_670ms.csv",
    ("P05", "arcex"): "tSCS_CHUV_data/24-07-2026/testSCS/Modulated_autosave_20260724_103530_508ms.csv",
    # P06's control blocks, cathodic, before lidocaine - the same recordings its polarity
    # notebook calls "before · cathodic". Thresholds are picked for the four TOP3 muscles only,
    # so its other muscles fall back to detection in the eight-muscle figures.
    ("P06", "burst"): "tSCS_CHUV_data/28-08-2026/Burst_autosave_20260928_152710_593ms.csv",
    ("P06", "arcex"): "tSCS_CHUV_data/28-08-2026/Modulated_autosave_20260928_154023_427ms.csv",
}
SUBJECTS  = ["P03", "P04", "P05", "P06"]
PROTOCOLS = ["burst", "arcex"]
NAME  = {"burst": "30 Hz burst", "arcex": "ARC-EX"}
SCOL  = {"P03": "#1f3b73", "P04": "#e6550d", "P05": "#2ca25f", "P06": "#756bb1"}

# the muscles every one of the three recordings has, minus the proximal ones
MUSCLES = ["Thenar (L)", "Ext. digitorum (L)", "Flex. digitorum (L)", "Flex. carpi rad. (L)",
           "Thenar (R)", "Ext. digitorum (R)", "Flex. digitorum (R)", "Flex. carpi rad. (R)"]

# ---- peak detection: the same settings as every other notebook ----------------------------
N_PULSES, RESP_START_MS, RESP_END_MS, GUARD_MS = 10, 8.0, None, 1.0
SNR_ON, MIN_SNR = "median", 1.2
ANCHOR, ANCHOR_WIN_MS = "mean", 3.0
MAX_EDGE_FRAC, EDGE_MS, JITTER_MS = 0.5, 1.0, 0.5
KW = dict(n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, resp_end_ms=RESP_END_MS,
          guard_ms=GUARD_MS, min_snr=MIN_SNR, max_edge_frac=MAX_EDGE_FRAC, snr_on=SNR_ON,
          anchor=ANCHOR, anchor_win_ms=ANCHOR_WIN_MS)
CONSECUTIVE = 2        # intensities the response criterion must hold for

for k, f in RUN.items():
    meta, _, sig = load_run(f)
    have = {pretty(c) for c in sig if c != "Trigger A"}
    amps = sorted({m["amp_ma"] for m in meta})
    print(f"{k[0]:4s} {NAME[k[1]]:12s} elec={meta[0]['electrode']}  "
          f"{amps[0]}-{amps[-1]} mA ({len(amps)} steps)  "
          f"missing: {sorted(set(MUSCLES) - have) or 'none'}")

TOP3 = ["Flex. digitorum (L)", "Ext. digitorum (L)",      # left arm, then right
        "Flex. digitorum (R)", "Flex. carpi rad. (R)"]
#   the muscles with a threshold in all six baseline recordings. Ext. digitorum (R),
#   Flex. carpi rad. (L) and both Thenars are missing one or more, so they are out.
#   Flex. carpi rad. (R) is in, but its P04 ARC-EX response sits AT the noise floor (1x), so
#   that one bar is blanked - the threshold there needs picking by hand.


## 2 · The motor thresholds you chose

Read straight from `results/<session>/mt_*.csv`, so this is what you actually picked — nothing
here can substitute a different value. One row per recording, one column per muscle: every
intensity stacked, **your threshold in red**, the mA in the corner. `not picked` = nothing saved
for that muscle in that recording; change it in section 5.

Each recording is analysed at **its own** threshold, so 30 Hz and ARC-EX are each read at the
current that muscle actually needed under that protocol.

In [ ]:
CHECK = {f"{s} · {NAME[p]}": RUN[(s, p)] for s in SUBJECTS for p in PROTOCOLS}

picked = fig_threshold_grid(CHECK, TOP3, title="the thresholds saved for each recording")

### The numbers, and where each one came from

In [ ]:
MT = {}      # MT[(subject, protocol)] = {muscle: mA}, picked in notebooks/motor_thresholds/
for s in SUBJECTS:
    for p in PROTOCOLS:
        csv = RUN[(s, p)]
        MT[(s, p)] = thresholds_for(csv, MUSCLES, consecutive=CONSECUTIVE, **KW)
        print(f"{s:4s} {NAME[p]:12s} {threshold_source(csv)}")

print(f"\nmotor threshold (mA)\n{'muscle':22s}"
      + "".join(f"{s + ' ' + p[:5]:>14s}" for s in SUBJECTS for p in PROTOCOLS))
for m in MUSCLES:
    print(f"{m:22s}" + "".join(f"{str(MT[(s, p)].get(m) or '-'):>14s}"
                               for s in SUBJECTS for p in PROTOCOLS))

MT_STAMP = mt_stamp(RUN)

## 3 · Are the peaks detected correctly? — every recording, every muscle

The same grid as section 2 — a row per recording, a column per muscle — **each at the threshold you picked for that
recording** (the red trace in section 2). 30 Hz and ARC-EX have different thresholds in the same
muscle — that is the result, not an error — so each is checked at its own. The 10 pulses are
re-aligned on their own onset, with the max (v) and min (^) the peak-to-peak is made of.

| | |
|---|---|
| **green** | the response window — `RESP_START_MS` after the pulse to `GUARD_MS` before the next |
| **red bands** | the only places a peak may be taken: `ANCHOR_WIN_MS` either side of the train's own average latency. **A peak outside these is skipped however clear it looks** |
| markers stacked | the same deflection found on every pulse |
| red ring | flagged — on a window border, or at a different latency from the rest |
| corner | `clean`, or how many of the 10 pulses were flagged |

If a whole train sits in the noise, a blank bar in section 4 is correct.

### If a peak you can see is not being taken

`TRY` changes the settings **for these figures only**, so you can test whether detection is too
strict without touching any result. When you find settings you want, copy them into `KW` in
section 1.

| setting | now | loosen it by | what it controls |
|---|---|---|---|
| `anchor_win_ms` | 3.0 | **raising** it, or `anchor=None` | how far from the train's average latency a peak may sit — much the most common reason a visible response is missed |
| `resp_start_ms` | 8.0 | **lowering** it | when the window opens after the pulse; too low and it takes the stimulus artifact |
| `min_snr` | 1.2 | **lowering** it, or `None` | how far above that muscle's noise a train must be to count at all |
| `max_edge_frac` | 0.5 | **raising** it, or `None` | how many pulses may peak on a window border before the whole train is called artifact |

In [ ]:
# ======== try different detection settings, for this figure only ========
TRY = dict(KW)                     # start from the analysis settings...
# TRY["anchor_win_ms"] = 6.0       # ...and loosen one at a time: widen the red bands
# TRY["anchor"] = None             # no anchoring: biggest max/min anywhere in the window
# TRY["resp_start_ms"] = 6.0       # open the window sooner after the pulse
# TRY["min_snr"] = None            # stop discarding trains that sit close to noise
# TRY["max_edge_frac"] = None      # stop discarding trains that peak on a window border
# ========================================================================

# same grid as section 2 - a row per recording, a column per muscle - but showing how the
# peaks are found at the threshold picked for THAT recording
MT_ROW = {f"{s} · {NAME[p]}": MT[(s, p)] for s in SUBJECTS for p in PROTOCOLS}

fig_detection_grid(CHECK, MT_ROW, TOP3, edge_ms=1.0, jitter_ms=0.5,
                   title="peak detection at each recording's own threshold", **TRY)

## 4 · The results

### The three participants in one plot

In [ ]:
# every recording normalised within itself: x in x MT, y in % of that muscle's max
CURVES = {k: recruitment(RUN[k], MUSCLES, MT[k], **KW) for k in RUN}
print(f"{len(CURVES)} recordings, {len(MUSCLES)} muscles")

In [ ]:
xref, at_ref = fig_across_subjects(CURVES, TOP3, SUBJECTS, protocols=("burst", "arcex"),
                                  colours=SCOL, names=NAME, band=True,
                                  title="Recruitment shape is the same across participants",
                                  save=None)   # save="figures/across_participants/three_subjects.png"

### One figure per muscle — the participants side by side

In [ ]:
# ======== settings ========
MIN_SNR_BAR = 2.0   # a bar is left blank if its 1st pulse is under this x its own noise -
                    # the ratio divides by that pulse, so a tiny one makes the bar meaningless.
                    # Lower it to show more bars, raise it to show only solid ones.
WITH_P1     = True  # add a first panel with the 1st pulse itself, in mV - the
                    # quantity the other two are a percentage OF. Compare it WITHIN
                    # a participant (30 Hz vs ARC-EX); mV do not carry between people.
REST_FROM   = 2     # the "mean of pulses N-10" panel starts here (2 = what we have been using)
# ==========================

warn_if_stale(RUN, MT_STAMP)

for m in TOP3:
    fig_bars(RUN, MT, m, SUBJECTS, protocols=("burst", "arcex"), steps=0,
             names=NAME, min_snr_ratio=MIN_SNR_BAR, rest_from=REST_FROM, with_p1=WITH_P1,
             title=m,
             save=None, **KW)   # save=f"figures/across_participants/bars_{m[:8]}.png"


### 4c · Across participants — how much the response falls along the train

**The figure to read first.** For each muscle: the pale bar is the **1st pulse**, which is 100 %
by construction and is the quantity the other two are a percentage *of*; next to it, **30 Hz** and
**ARC-EX**, each as the mean of pulses **2–10** as a % of their own 1st pulse. So the height of the
grey and red bars against the pale one is the drop, and **100 % = no drop at all**.

The bar is the **mean over participants**, the error bar the **SD** (set `ERR = "sem"` for
standard error), and every participant is a **white dot**. `n` sits on each bar because the noise
filter can take different participants from the two protocols.

The second figure is the same thing for the **2nd pulse alone** — the pulse 33 ms after the first,
which is the interval a posterior-root reflex should be most depressed at. See section 6.

Each train is measured against **itself**, so a participant is included even where the two
protocols are not comparable in millivolts. A train whose 1st pulse is under `MIN_SNR_BAR` x its
own noise is left out and named underneath, because these panels divide by that pulse.

**To add a participant**, put them in `RUN` and `SUBJECTS` in section 1; the bars, the error bars
and every `n` follow.

In [ ]:
# each protocol is its own group: they sit ~70 mA apart and never share an intensity
GRP  = {p: p for p in PROTOCOLS}
dep  = condition_table(RUN, MT, TOP3, PROTOCOLS, base=GRP, group=GRP, match="common", **KW)

# ======== settings ========
ERR = "sd"        # "sd" = standard deviation across participants | "sem" = standard error
# ==========================

for key, lab, ttl in (
        ("dep_rest", "% of the 1st pulse",
         "Cathodic baseline — mean of pulses 2–10, % of the 1st pulse"),
        ("dep2", "% of the 1st pulse",
         "Cathodic baseline — 2nd pulse (33 ms), % of the 1st pulse")):
    fig_group_summary(dep, SUBJECTS, TOP3, PROTOCOLS, key=key,
                      colours={p: c for p, c in zip(PROTOCOLS, ("#9A9A9A", "#C0392B"))},
                      labels=NAME, min_snr_ratio=MIN_SNR_BAR, err=ERR,
                      reference=("1st pulse", 100),   # the quantity the others are a % OF
                      ylabel=lab, ref=100, ylim=(0, 175), title=ttl,
                      save=None)   # save=f"figures/across_participants/{key}.png"

## 5 · Change a threshold

One cell per recording: move the slider, press **Set to slider**, then **Save**. Re-run section 2
and every figure follows.

In [ ]:
def repick(key):
    """Open the picker for one recording of this comparison, with a Save button.

    The green line is the DETECTOR's answer for this recording - not the baseline, and not
    whatever is already saved (that is the orange trace)."""
    csv = RUN[key] if isinstance(RUN, dict) and key in RUN else runs[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
repick(('P03', 'burst'))

In [ ]:
repick(('P03', 'arcex'))

In [ ]:
repick(('P04', 'burst'))

In [ ]:
repick(('P04', 'arcex'))

In [ ]:
repick(('P05', 'burst'))

In [ ]:
repick(('P05', 'arcex'))

---

## 6 · What this notebook shows, and what it cannot tell us

### The result

**ARC-EX needs about 2.9x the current of 30 Hz to reach the same muscle's motor threshold**, in
all four participants — per-participant means **2.77 (P03), 3.06 (P04), 3.01 (P05), 2.83 (P06)**,
and **2.91 +- 0.29 over the 15 muscle-wise ratios**. That is the most reproducible thing in the
dataset: it survived adding a fourth participant, it does not depend on which threshold rule is
used, and the spread within a participant is as large as the spread between them.

**Above threshold the two protocols recruit the arm the same way.** The recruitment curves, once
each is put on its own x MT scale, sit on top of each other, and no muscle is selectively
favoured by one protocol.

**Along the train they differ, and not consistently.** ARC-EX depresses more than 30 Hz in most
muscles of P03 and P04; P05 is too close to noise for most of its muscles to say.

Millivolts are never compared between participants — different electrodes, skin and gain. Every
statement above is either a ratio within a participant, or a comparison of such ratios.

### The question this notebook raises but does not answer

At 30 Hz the interval between pulses is **33 ms**, which is the interval used to tell a
**posterior-root reflex** from **direct activation of motor axons**:

- a reflex is strongly post-activation depressed at 33 ms — the 2nd response falls well below half
- a direct motor response follows 1:1 and is not suppressed

So the "2nd pulse" panel above is, unintentionally, that test. The cell below prints it with the
onset latency beside it.

In [ ]:
# the 33 ms paired-pulse test hidden inside every 30 Hz train
import warnings as _w
from functions.burst import burst_p2p, noise_p2p

print(f"{'':18s}{'muscle':22s}{'mA':>5s}{'x noise':>9s}{'latency ms':>12s}"
      f"{'2nd pulse %':>13s}{'mean 2-10 %':>13s}")
for s in SUBJECTS:
    for p in PROTOCOLS:
        csv = RUN[(s, p)]
        meta_, t_, sig_ = load_run(csv)
        chans = [c for c in sig_ if c != "Trigger A"]
        res = burst_p2p(meta_, t_, sig_, chans, **{**KW, "min_snr": None, "max_edge_frac": None})
        amps = list(res["amps"])
        nbA = noise_p2p(t_, sig_, chans, win_len_ms=res["win"][0][1] - res["win"][0][0])
        for m in TOP3:
            ch = next((c for c in chans if pretty(c) == m), None)
            th = MT[(s, p)].get(m)
            if ch is None or not th or th not in amps:
                continue
            w = amps.index(th)
            y = np.asarray(res["p2p"][ch][w], float)
            nb = np.asarray(nbA[ch], float); n = float(nb[w]) if nb.ndim else float(nb)
            if not n or y[0] / n < MIN_SNR_BAR:      # same guard as the bars above
                continue
            at = res.get("anchor_t", {}).get(ch)
            lat = (np.nanmean(at[w]) - res["pulse_ms"][0]
                   if at is not None and np.isfinite(at[w]).any() else np.nan)
            with _w.catch_warnings():
                _w.simplefilter("ignore", RuntimeWarning)
                print(f"{s + '  ' + NAME[p]:18s}{m:22s}{th:5g}{y[0]/n:9.1f}{lat:12.1f}"
                      f"{100*y[1]/y[0]:13.0f}{100*np.nanmean(y[1:])/y[0]:13.0f}")

### How to read that table

**Most 2nd pulses come out near 100 %.** If these responses were monosynaptic posterior-root
reflexes, they should be strongly suppressed at 33 ms and they are not. Taken at face value that
points to a **substantial direct component** — motor axons in the ventral roots driven directly —
rather than the pure reflex that tSCS work usually assumes.

Three things argue the other way, so this is not a conclusion:

| | |
|---|---|
| **latency 10–24 ms** | direct motor root to forearm is roughly 8–12 ms; the longer values leave room for a synapse |
| **pulses 2–10 fall to 45–90 %** | something is depressing across the train, which pure motor-axon activation should not do at 30 Hz |
| **the pattern is muscle-specific** | the longest-latency muscles depress most, which is what a *mixed* response looks like — an early direct component and a later reflex one, in different proportions per muscle |

Everything here is measured **at threshold**, where the response has only just appeared. The
mixture can change with intensity, so none of this transfers automatically to the intensities used
for a protocol.

### What would settle it, in order

1. **Paired pulse at 33 and 100 ms** — the proper version of the test above, with no ongoing train
   to confound it. `Paired_Pulse_*` files already exist for P03, P04 and P05.
2. **Recruitment-curve shape past threshold** — a reflex grows, peaks, then declines as the
   antidromic motor volley collides with it; direct activation grows to a plateau. The sweeps in
   this notebook already contain this.
3. **Onset latency from the single pulses**, against the motor conduction time predicted for that
   arm. `Single_Pulse_*` files and the latency picker already exist.
4. **The vibration blocks, re-read as a Ia test** — tendon vibration presynaptically inhibits Ia
   terminals, so a Ia-mediated reflex should be suppressed at the vibrated muscle. In
   `P03_P04_tendon_vibration.ipynb` it was not, which is weak evidence against a Ia reflex once
   the block-to-block drift is accounted for.
5. Only if 1–4 disagree: a **background voluntary contraction** block (facilitates a reflex, not a
   direct response) or a **frequency series**.

None of 1–4 needs a new recording.